# GTAN — Gated Temporal Attention Network (Colab GPU)

Trains the production DL model (`train_gtan.py`) as a **5-seed deep ensemble** on the MIMIC-IV cardiac-progression arrays, on a **free T4 GPU**.

**Steps**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Run cell 1, click **Choose Files**, upload `gtan_bundle.zip` (from the repo's `colab/` folder).
3. Run cells 2–4. Training takes ~8–12 min on a T4.
4. Cell 5 zips the results; download `gtan_outputs.zip` and drop its contents back into the repo (`data/preprocessed/improved_probs/` and `results_improved.json`).

In [ ]:
# Cell 1 — upload the bundle and unpack it into the layout train_gtan.py expects
import os, zipfile, shutil
from google.colab import files

up = files.upload()  # choose gtan_bundle.zip
name = next(iter(up))

ROOT = '/content/proj'
DATA = f'{ROOT}/data/preprocessed'
SRC  = f'{ROOT}/src/model_training'
for d in (DATA, SRC):
    os.makedirs(d, exist_ok=True)

with zipfile.ZipFile(name) as z:
    z.extractall('/content/_bundle')

for f in os.listdir('/content/_bundle'):
    src = f'/content/_bundle/{f}'
    if f.endswith('.py'):
        shutil.copy(src, f'{SRC}/{f}')
    else:
        shutil.copy(src, f'{DATA}/{f}')
print('data:', sorted(os.listdir(DATA)))
print('src :', sorted(os.listdir(SRC)))

In [ ]:
# Cell 2 — confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (set Runtime->GPU!)')

In [ ]:
# Cell 3 — import the repo's train_gtan.py and point its paths at /content/proj
import sys
from pathlib import Path
sys.path.insert(0, '/content/proj/src/model_training')

import train_gtan as G
G.ROOT     = Path('/content/proj')
G.DATA_DIR = Path('/content/proj/data/preprocessed')
G.MODEL_DIR = Path('/content/proj/models/gtan')
G.PROB_DIR = Path('/content/proj/data/preprocessed/improved_probs')
print('Using DATA_DIR =', G.DATA_DIR)
print('Model source is the SAME train_gtan.py as the local repo — no divergence.')

In [ ]:
# Cell 4 — run the 5-seed deep ensemble on GPU by calling the script's own main()
import sys
sys.argv = ['train_gtan.py',
            '--seeds', '5',
            '--epochs', '120',
            '--patience', '20',
            '--batch_size', '256',
            '--loss', 'focal',
            '--d_model', '64',
            '--log_every', '10']
G.main()

In [ ]:
# Cell 5 — package the outputs for download
import zipfile, os
out = '/content/gtan_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    pd = '/content/proj/data/preprocessed/improved_probs'
    for f in ('gtan_val_cal.npy', 'gtan_test_cal.npy'):
        p = os.path.join(pd, f)
        if os.path.exists(p):
            z.write(p, arcname=f'improved_probs/{f}')
    rj = '/content/proj/results_improved.json'
    if os.path.exists(rj):
        z.write(rj, arcname='results_improved.json')
    md = '/content/proj/models/gtan'
    if os.path.isdir(md):
        for f in os.listdir(md):
            z.write(os.path.join(md, f), arcname=f'models_gtan/{f}')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)